# Semi-Supverised Graph Attention Network

Info:

Please run once with your data then change the passs_grade in data.csv to the grade you want
If training on GPU (recommended) change decide to 'device = torch.device('cuda')'

Imports and File Paths

In [ ]:
# Run this ONCE first 
!pip install torch 
!pip install torch_geometric

In [ ]:

import pandas as pd
import numpy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

## Import Moudles
from torch_geometric.nn import GATv2Conv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

device = torch.device('cpu') # Use this CPU for running the model


path_1 = '/home/c3646202/Desktop/FootballPassingAnalysisProject2/output/data.csv' # demo1 dataset

passes_csv_path = '/home/c3646202/Desktop/FootballPassingAnalysisProject2/pass_data_processing/passes.csv' ## Passes demo1 dataset





# Graph Construction

Node - Each node represent a player or a ball with features:

Nodes Features - player_x, player_y, team, has_ball, ball_x, ball_y

Edges - Each edge is the interaction between players:

Edges Features - Interaction between players - Distance_to_ball. Gives context to edge about the relationship beween the players - has_ball, team


Edge Index - This connect each player to every other player in the graph

Edge Attributes - Edge Features

Graph Level Prediction - 1 graph per pass

Helped build the graph:
https://www.datacamp.com/tutorial/comprehensive-introduction-graph-neural-networks-gnns-tutorial?
https://www.geeksforgeeks.org/dsa/window-sliding-technique/
https://www.kaggle.com/code/rafsunsheikh/convert-any-tabular-data-to-graph-for-gnn
https://pytorch-geometric.readthedocs.io/en/2.5.2/get_started/introduction.html#data-transforms
https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html#exercises

In [ ]:

#------------------------------- 
#          Load Data In & CSV_Import
#-------------------------------

def csv_import(path):
    dataframe = pd.read_csv(path) # Read file into dataframe using Pandas
    return dataframe

# Loading passes data
passes_dataframe = pd.read_csv(passes_csv_path) # read data in
data_dataframe = pd.read_csv(path_1) 
y_grade = passes_dataframe[['pass_grade']] ## get pass_grade column
pass_graph = []

#------------------------------- 
#          Build Graph 
#-------------------------------


for index, rows in passes_dataframe.iterrows(): # creating the graph by going through each frame
    start_frame = rows['start_frame']
    end_frame = rows['end_frame']
    row = []
    column = []

    #frame - Refrance - https://www.geeksforgeeks.org/python/how-to-fix-valueerror-the-truth-value-of-a-series-is-ambiguous-in-pandas/ 
    # # validating if start and end frame are not before or after each other
    frame = data_dataframe[(data_dataframe['frame'] >= start_frame) & (data_dataframe['frame'] <= end_frame)] 
    if frame.empty: # check if fram is empy
        continue

    node_features = frame[['player_x', 'player_y', 'team', 'has_ball', 'ball_x', 'ball_y']]  ## Node Features for that create the node
    edge_features = frame[['distance_to_ball', 'has_ball', 'team']]

    #team = frame['team'].values
    frame_length = len(frame)
    for i in range(frame_length):   ### Works for now, can be slow and can refactored
        for j in range(frame_length):
            if i != j: ## i !=j = connects all player | for just teammates- if i != j and team[i] == team[j]:  
                ## this is only ball holder to temmates, need teammate, might need to change if I want cross team interaction can test
                row.append(i)
                column.append(j) 
    #   Data Formatting for Graph Attention Network 
    node_features = node_features.values.astype('float32') ## convertz float32 to pytorch
    edge_features = edge_features.values[column].astype('float32') 
    edge_index = torch.tensor([row, column], dtype=torch.long) # Converting to tensor
    x = torch.tensor(node_features)
    
    if pd.isna(rows['pass_grade']): ## Skips passes without a gradehttps://www.w3schools.com/python/pandas/ref_df_isna.asp
        continue
    percentage_to_grade = {0.1: 0, 0.3: 1, 0.5: 2, 0.7: 3, 0.9: 4} ## model only takes in whole numbers
    grade_value = percentage_to_grade[rows['pass_grade']]
    y_grade = torch.tensor([grade_value] , dtype=torch.long) # from 0-4 to 1-5 Refrance
    edge_attr = torch.tensor(edge_features)
    
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y_grade=y_grade,) #edge_attr=edge_attr
    pass_graph.append(data)
    
    #  Refrance - Data() - https://pytorch-geometric.readthedocs.io/en/2.5.2/get_started/introduction.html#data-transforms





# Graph Attention Network - GATv2

The model has been adpated to tailour to our requirement from - https://github.com/pyg-team/pytorch_geometric/blob/master/examples/gat.py

Paramaters:

Nodes - node features
Nodes features – Player features and ball features, such as 
Edge features – Distance to the ball, distance to nearest player, ball_speed
Edge_index – Row and columns of the 
Edge_attr - redg

Layers:

Layer 1:
Player's Atrtributes 

Layer 2:
Player to Player Connections

Layer 3:
Team to Team Connections

Surpporting Links:

https://distill.pub/2021/gnn-intro/
geeksforgeeks.org/deep-learning/what-is-layer-normalization/
https://distill.pub/2021/understanding-gnns/
https://pytorch-geometric.readthedocs.io/en/2.7.0/modules/nn.html
https://github.com/recodehive/machine-learning-repos
https://www.geeksforgeeks.org/blogs/machine-learning-pipeline/
https://halil7hatun.medium.com/graph-neural-networks-gnns-1f463df4bb77





In [ ]:


loader = DataLoader(pass_graph, batch_size= 4, shuffle= False)  # Loading in the data from the build graph
## Refrance Loder - https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
##----------------------------------------------------------

#----------------------------------
#             Config
#----------------------------------

in_channels = 6 # number of nodes
hidden_channels  = 64   # hiddel size for GAT layers
Dropout= 0.2 # probability
per_head = hidden_channels // 2 # hidden units per head
num_grade_classes = 5 # Classes for pass grade blunder, mistake, textbook, good, brilliant
compress_channel = 32 ## hidden channel compressed
export_data_model_path = '/home/c3646202/Desktop/FootballPassingAnalysisProject2/models/model.pt'



#----------------------------------
#             Model
#----------------------------------
# Model Refrance -# This template code is used from -  https://github.com/pyg-team/pytorch_geometric/blob/master/examples/gat.py 
# supporting # https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html

class PassingGAT(nn.Module):
    def __init__(self):
        super().__init__()
        ## Layer 1 - Player Attributes
        #Refrance for GATv2Conv - https://pytorch-geometric.readthedocs.io/en/2.7.0/generated/torch_geometric.nn.conv.GATv2Conv.html
        self.conv1 = GATv2Conv(in_channels, per_head , heads=2, edge_dim=3, concat=True, dropout=Dropout) 
        # Stabilises taining using Normalization
        self.norm1 = nn.LayerNorm(hidden_channels)
        # Layer 2 - Players to Player Connections
        self.conv2 = GATv2Conv(hidden_channels, per_head, heads=2, edge_dim=3, concat=True, dropout=Dropout)
        self.norm2 = nn.LayerNorm(hidden_channels)
         # Layer 3 - Team to Team Connections
        self.conv3 = GATv2Conv(hidden_channels, hidden_channels, heads=1, edge_dim=3, concat=False, dropout=Dropout)

        # Residual projection - ensures input feature added to the output
        self.res_proj = nn.Linear(in_channels, hidden_channels)

        self.grade_head   = nn.Linear(hidden_channels, num_grade_classes)
 
    def forward(self, x, edge_index, batch, edge_attr):

        # Residual connection
        res = self.res_proj(x)

        # Layer 1 - Player's Atrtributes 
        x = self.conv1(x, edge_index, edge_attr)
        x = F.elu(x)
        x = self.norm1(x)
        x = F.dropout(x, p=Dropout, training=self.training)
        # Layer 2 - Player to Player Connections
        x = self.conv2(x, edge_index, edge_attr)
        x = F.elu(x)
        x = self.norm2(x)
        x = F.dropout(x, p=Dropout, training=self.training)
        # Layer 3 - Team to Team Connections
        x = self.conv3(x, edge_index, edge_attr)

        # orignal + results
        x = x + res ## Could break if res_proj dimention of model change




        #Reconsruction of the data
        x = global_mean_pool(x, batch) # from 40, 64 to  1, 64 one grade per pass
       # predict
        grade = self.grade_head(x)

        return grade # outcome  # could use cross entropy loss but use this and nll_
device = torch.device('cpu')   
model = PassingGAT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4) # lr stands for learning rate

#------------------------------- 
#           Train
#-------------------------------

def export_data(model):
    torch.save(model.state_dict(), export_data_model_path) ### Refrance https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
    #grades.to_csv('ML_Results.csv', index=False)

##  Train
def train(data, model, device):
    model.train() # train mode
    optimizer.zero_grad() # reset gradients
    grade = model(data.x, data.edge_index, data.batch, data.edge_attr) 
    # https://stackoverflow.com/questions/69965519/cross-entropy-loss-argument-target-position-2-must-be-tensor-not-numpy-n
    print(data.x.shape)
    grade_loss = F.cross_entropy(grade, data.y_grade.view(-1)) ## difference between actual and predicted grade
    ## refrance - . view() https://discuss.pytorch.org/t/valueerror-expected-input-batch-size-1-to-match-target-batch-size-4-how-can-i-fix-this/202084
    loss = grade_loss
    # Backpropagation
    loss.backward() 
    optimizer.step()
    return float(loss.detach()) # need to change back to return total_loss.detech() or item()


#------------------------------- 
#            Test
#-------------------------------

grades_data = []
@torch.no_grad()  ## do we need this
def test(loader, model, device):
    device = torch.device('cpu')
    model.eval()
    accs = []   
    for data in loader:
        data = data.to(device)
        grade = model(data.x, data.edge_index, data.batch, data.edge_attr) #https://halil7hatun.medium.com/graph-neural-networks-gnns-1f463df4bb77
        grade_predictions = grade.argmax(dim=1)
        # Compare the grades
        correct_predictions = int((grade_predictions == data.y_grade).sum()) ##
        total = data.y_grade.shape[0]
        ## Accuarcy of training
        accs.append(correct_predictions/total)

    return accs


#------------------------------- 
#          Training Loop
#-------------------------------
 
def training(loader, model, device, epoch=100):
    times = []
    for epoch in range(1, epoch + 1): ## train the model depeding on iteration first round 100 second 80
        start = time.time()
        for data in loader:
            data = data.to(device)
            loss = train(data, model, device)
            print(f'loss: {loss}')
        accs = test(loader, model, device)
        print(f'grade : {accs}')
        times.append(time.time() - start)
    return times, accs, loss


def display_ouput(times, accs, epoch, loss, model):
    print(f"Median time per epoch: {torch.tensor(times).median():.4f}s")
    print(f'Grade Accuracy: {accs} Epoch: {epoch}, Loss: {loss}')
    print(f'\n===================================================================\n \t\tExporting \n===================================================================\n') 
  #  print(f'\tPassing Grade Results:\n\n{grades}\n\n')
    print(f'\tModel:\n\n{model}\n \n===================================================================\n')
    export_data(model) 

def main():
    model.to(device)  ## sending the model to th GPU or CPU
    times, accs, loss = training(loader, model, device, epoch=100) 
    display_ouput(times, accs, 100, loss, model)

main()


 


In [ ]:
# Run 2
##https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
load_model = torch.load('model.pt')
model.load_state_dict(load_model['model_state_dict'])


#model = PassingGAT().to(device)
#model.load_state_dict(torch.load('model.pt', weights_only=True))
#model.eval()